# NutriGo · Week 3 — Food recognition: transfer learning on Food-101

**Run on Kaggle** with GPU (T4/P100) — about 1–2 hours.
Add the dataset `dansbecker/food-101` as an input.

Pipeline: photo 224×224 → augment (flip · rotate · zoom · contrast) → EfficientNetB0 (ImageNet) → global average pool + dropout 0.3 → softmax 101

| Phase | What trains | Epochs | Learning rate |
|---|---|---|---|
| 1 · Head | new classifier only (base frozen) | 4 | 1e-3 |
| 2 · Fine-tune | top 60 EfficientNet layers + head | 8 | 1e-4 (reduce on plateau) |
| Setup | 80/20 split · batch 64 · mixed precision · early stopping | | Adam |

Outputs (in `/kaggle/working/`): `food101_model.keras`, `labels.txt`, `results.json`, curves.
Copy the first two into `backend/models/`.

In [ ]:
import glob, json, os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

IMG, BATCH, SEED = 224, 64, 42
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

## 1 · Find the images folder (101 class sub-folders)

In [ ]:
def find_images_dir(root='/kaggle/input'):
    for d in glob.glob(os.path.join(root, '**', 'images'), recursive=True):
        subs = [s for s in os.listdir(d) if os.path.isdir(os.path.join(d, s))]
        if len(subs) == 101:
            return d
    raise FileNotFoundError('Add the Food-101 dataset as a Kaggle input')

IMAGES = find_images_dir()
print(IMAGES)

In [ ]:
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    IMAGES, validation_split=0.2, subset='both', seed=SEED,
    image_size=(IMG, IMG), batch_size=BATCH, label_mode='int')
class_names = train_ds.class_names
print(len(class_names), 'classes |', class_names[:5])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 2 · Model: EfficientNetB0 + new head

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='augment')

base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
base.trainable = False  # phase 1

inputs = tf.keras.Input((IMG, IMG, 3))
x = augment(inputs)
x = base(x, training=False)          # EfficientNet rescales 0-255 internally
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax', dtype='float32')(x)
model = tf.keras.Model(inputs, outputs, name='nutrigo_effnetb0')

metrics = ['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')]
model.summary(show_trainable=True)

## 3 · Phase 1 — train the head (base frozen)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=metrics)
early = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=4, callbacks=[early])

## 4 · Phase 2 — fine-tune the top 60 layers

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:            # keep BatchNorm frozen for stable fine-tuning
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=metrics)
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=1, min_lr=1e-6),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
]
start = len(hist1.history['loss'])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=start + 8, initial_epoch=start, callbacks=callbacks)

## 5 · Evaluation: top-1, top-5, per-class F1

In [ ]:
loss, top1, top5 = model.evaluate(val_ds)
print(f'Top-1 accuracy: {top1:.1%} | Top-5 accuracy: {top5:.1%}')

y_true, y_pred = [], []
for xb, yb in val_ds:
    y_pred.append(np.argmax(model.predict(xb, verbose=0), axis=1)); y_true.append(yb.numpy())
y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

rep = pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, output_dict=True)).T
per_class = rep.iloc[:len(class_names)].sort_values('f1-score')
print('Hardest 10 dishes'); display(per_class.head(10).round(3))
print('Easiest 5 dishes'); display(per_class.tail(5).round(3))

cm = confusion_matrix(y_true, y_pred); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], class_names[i], class_names[j]) for i in range(len(cm)) for j in range(len(cm)) if cm[i, j]), reverse=True)[:10]
print('Most confused pairs (true -> predicted):')
for n, a, b in pairs:
    print(f'  {a:25s} -> {b:25s} {n}')

## 6 · Accuracy and loss curves (both phases)

In [ ]:
h = {k: hist1.history[k] + hist2.history[k] for k in hist1.history}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for key, a in (('accuracy', ax[0]), ('loss', ax[1])):
    a.plot(h[key], label='train'); a.plot(h['val_' + key], label='validation')
    a.axvline(len(hist1.history['loss']) - 0.5, color='grey', ls='--', label='fine-tune starts')
    a.set_title(key); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/training_curves.png', dpi=150); plt.show()

## 7 · Save the model for the Flask API

In [ ]:
model.save(f'{OUT}/food101_model.keras')
with open(f'{OUT}/labels.txt', 'w') as fh:
    fh.write('\n'.join(class_names))
results = {'top1': round(float(top1), 4), 'top5': round(float(top5), 4),
           'val_images': int(len(y_true)), 'vit_nateraw_food_reported_top1': 0.891,
           'hardest': per_class.head(10)['f1-score'].round(3).to_dict()}
json.dump(results, open(f'{OUT}/results.json', 'w'), indent=2)
print(json.dumps(results, indent=2))
print('Download food101_model.keras + labels.txt -> backend/models/')

## 8 · Compare with the pretrained ViT

| Model | Top-1 | Top-5 |
|---|---|---|
| ViT `nateraw/food` (reported) | 89.1% | — |
| Our EfficientNetB0 | *fill from results.json* | *fill* |

Put our top-1 / top-5 on slide 7 ("Our EfficientNet ___%").